# Retrieval-Augmented Generation (RAG) Pipeline

This notebook demonstrates the complete RAG pipeline.

The pipeline is divided into two major phases:

1. **Document Ingestion and Indexing**
   - Read the document
   - Preprocess the extracted text
   - Create chunks
   - Generate embeddings
   - Store the embeddings in a vector store

2. **Query and Response Generation**
   - Receive a user query
   - Generate the query embedding
   - Retrieve relevant document chunks
   - Build a prompt using the query and retrieved context
   - Generate the final response using an LLM

# Phase 1: Document Ingestion and Indexing

In this phase, the source document is transformed into searchable vector representations.

The document passes through the following pipeline:

```PDF
↓
Text Extraction
↓
Text Preprocessing
↓
Chunking
↓
Embedding Generation
↓
Vector Store

## 1. PDF Reading and Markdown Extraction

The first step is to extract the content from the PDF document.

The document is initially converted into a markdown representation.

In [1]:
import pymupdf
import pymupdf4llm
from src.preprocessing.ingestion_preprocessing import HeaderInfoBuilder
FILE_PATH ="./src/pdfs/logical_reasoning.pdf"

In [2]:
header_builder = HeaderInfoBuilder()
hdr_info = header_builder.build(FILE_PATH)

In [3]:
pymupdf4llm.use_layout(False)

doc = pymupdf.open(FILE_PATH)

markdown = pymupdf4llm.to_markdown(
    doc,
    show_progress=True,
    # use_ocr=False,
    hdr_info=hdr_info
)


Processing ./src/pdfs/logical_reasoning.pdf...


100%|██████████| 239/239 [00:19<00:00, 12.29it/s]


## 2. Text Preprocessing

The raw text extracted from a PDF may contain structural noise such as:

- Repeated headers
- Repeated footers
- Page numbers
- Other boilerplate content

The purpose of this stage is to identify and remove repeated structural content while preserving the actual semantic content of the document.

The preprocessing pipeline uses similarity analysis and statistical patterns to identify boilerplate content dynamically rather than relying on document-specific hardcoded strings.

In [4]:
"""
Markdown Preprocessing is pending and below code is written to preprocess the text file.
so it is commented out.
"""
# from src.preprocessing.cleaner import normalize_pdf
# from src.preprocessing.similarity import compute_line_similarity
# from src.preprocessing.boilerplate import identify_boilerplate, remove_boilerplate

'\nMarkdown Preprocessing is pending and below code is written to preprocess the text file.\nso it is commented out.\n'

In [5]:
# # Phase 1: PDF Text normalization
# normalized_pdf = normalize_pdf(pdf)

# # # Phase 2: Statistical Analysis
# similar_line_analysis = compute_line_similarity(pdf=normalized_pdf, candidate_lines=3)

# # Phase 3: Boilerplate Identification
# candidate_boilerplates = identify_boilerplate(similarity_df=similar_line_analysis)

# # print(candidate_boilerplates)

# # # Phase 4: Boilerplate Removal
# boilerplate_config = (
#     candidate_boilerplates[
#         candidate_boilerplates["is_boilerplate"]
#     ][["region", "line"]]
# )

# filtered_pdf = remove_boilerplate(normalized_pdf, boilerplate_config=boilerplate_config)

# # Phase 5: Converting page-wise PDF into the one combined text
# text = "".join(filtered_pdf)

In [6]:
# Markdown -> Tree
from src.preprocessing.markdown_parser import build_tree

markdown_tree = build_tree(text=markdown)

## 3. Text Chunking

The cleaned document is divided into smaller segments called chunks.

Chunking is necessary because the entire document cannot be efficiently embedded and retrieved as a single unit.

The objective of chunking is to create meaningful units of text that:

- Preserve sufficient semantic context
- Are small enough for efficient embedding and retrieval
- Can be independently retrieved in response to a user query

### Chunking — V1

The chunking process is performed in two layers.

#### 1. Semantic Chunking

First, we traverse the Markdown tree and create chunks based on the document's semantic hierarchy.

Each node that contains content becomes a chunk candidate.

For every chunk, we keep:

- `title`
- `content`
- `breadcrumb`

For parent nodes, the content also includes the node's subtopics.

The purpose of this layer is to preserve the **meaning and hierarchy** of the document.

> Semantic chunking answers: **"Where does the content naturally belong?"**

In [7]:
from src.chunking.semantic_chunking import semantic_chunker

In [8]:
chunks = semantic_chunker([markdown_tree])

In [9]:
# Verify
chunks[2:]

[{'title': '**a) What is Logical Reasoning?**',
  'content': 'Logical reasoning refers to the process of using a structured, methodical approach to arrive at conclusions\nor solve problems based on given premises or evidence. It is the foundation of critical thinking, enabling\nindividuals to analyze information, deduce conclusions, solve problems, and make decisions based on logic\nrather than assumptions or emotions.',
  'breadcrumb': ['',
   '**NOTES ON:** **Logical Reasoning and** **Data Interpretation**',
   '**A. Introduction to Logical Reasoning & Data** **Interpretation**',
   '**a) What is Logical Reasoning?**']},
 {'title': '**b) Importance of Logical Reasoning**',
  'content': 'Logical reasoning is crucial in every aspect of life. It allows individuals to:\n\n\n✓ **Make Informed Decisions:** By evaluating information logically, individuals can make better, more\n\ninformed decisions.\n\n\n✓ **Solve Problems Effectively:** Logical reasoning helps identify the root causes of p

#### 2. Recursive Text Splitter

The semantic chunks may still be too large for embedding.

Therefore, we pass the content of each semantic chunk through the `RecursiveCharacterTextSplitter`.

This layer only handles **size control**. It splits a large semantic chunk into smaller pieces while trying to preserve the local context.

> Recursive splitting answers: **"Is this semantic chunk small enough to process?"**

#### Pipeline

Markdown Tree  
↓  
**Semantic Chunking** → meaningful, hierarchy-aware chunks  
↓  
**Recursive Text Splitting** → size-controlled chunks  
↓  
Final chunks for embedding

In [10]:
from src.chunking.chunker import text_splitter

In [11]:
final_chunks = []
for chunk in chunks:
   splitted_chunks =  text_splitter(chunk['content'])

   final_chunks.append({
      'title': chunk['title'],
      'chunks': splitted_chunks,
      'breadcrumb': chunk['breadcrumb']
   })

Split PDF text into 4 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 5 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 2 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 2 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
Split PDF text into 1 sub-documents.
S

In [12]:
final_chunks[7:10]

[{'title': '**f) Conclusion**',
  'chunks': ['Logical reasoning is a fundamental skill that enhances our ability to think critically, solve problems, and\nmake informed decisions. By understanding its types, importance, and applications, and actively working\nto improve our logical reasoning skills, we can better navigate the complexities of the world around us.\nWhether through formal study, practical application, or engaging in activities that challenge our cognitive\nprocesses, enhancing our logical reasoning capabilities is a lifelong journey that pays dividends in every\naspect of our lives.\n\n\nPage **4** of **239**'],
  'breadcrumb': ['',
   '**NOTES ON:** **Logical Reasoning and** **Data Interpretation**',
   '**A. Introduction to Logical Reasoning & Data** **Interpretation**',
   '**f) Conclusion**']},
 {'title': '**A.1 Verbal and Non-verbal Logical Reasoning**',
  'chunks': ["Verbal and non-verbal reasoning are two distinct types of cognitive assessments used to evaluate dif

## 4. Document Embedding Generation

Each text chunk is converted into a numerical vector representation called an embedding.

The embedding represents the semantic meaning of the chunk in a vector space.

Semantically similar chunks should have similar vector representations.

The result of this stage is:

Text Chunks
        ->
Embedding Model
        ->
Vector Representations

In [13]:
from src.embeddings.embedder import chunk_encoder

In [14]:
# Prepare text for embedding while preserving metadata
embedding_texts = []
embedding_metadata = []

for chunk in final_chunks:
    for text in chunk["chunks"]:
        embedding_texts.append(text)

        embedding_metadata.append({
            "title": chunk["title"],
            "breadcrumb": chunk["breadcrumb"],
            "content": text
        })

# Generate embeddings
embeddings = chunk_encoder(embedding_texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Each embedding has 384 dimensions
Similarity matrix:
[[0.99999976 0.36035573 0.08945025 ... 0.8324391  0.8258854  0.8347509 ]
 [0.36035573 1.         0.46996588 ... 0.35655612 0.44603354 0.4076033 ]
 [0.08945025 0.46996588 1.0000001  ... 0.15610364 0.20110664 0.14385897]
 ...
 [0.8324391  0.35655612 0.15610364 ... 1.0000002  0.9607304  0.9357084 ]
 [0.8258854  0.44603354 0.20110664 ... 0.9607304  1.0000001  0.9396139 ]
 [0.8347509  0.4076033  0.14385897 ... 0.9357084  0.9396139  1.0000001 ]]


## 5. Vector Store

The generated embeddings are stored in a vector index.

The vector store is responsible for:

- Storing document embeddings
- Maintaining the relationship between vectors and their original text chunks
- Searching for vectors that are most similar to a query embedding

In this project, FAISS is used as the vector search backend.

The document ingestion pipeline is now complete:

PDF
->
Text Extraction
->
Preprocessing
->
Chunking
->
Embedding
->
Vector Store

In [15]:
from src.vector_store.vector_store import FaissVectorStore

In [16]:
# Configure FaissVectorStore
dimension = embeddings.shape[1]

vectorStore = FaissVectorStore(dimension=dimension)

In [17]:
# Adding Embeddings Into the Vector Store
vectorStore.add(embeddings=embeddings, metadata_list=embedding_metadata)

In [18]:
# Checking total Indexes
vectorStore.total_vectors()

244

# Phase 2: Query and Response Generation

The second phase begins when a user submits a question.

The query is transformed into the same vector space as the document chunks so that semantically relevant information can be retrieved.

The pipeline is:

User Query
->
Query Embedding
->
Vector Search
->
Relevant Context
->
Prompt Construction
->
LLM Response Generation

## 1. User Query

The user provides a natural-language question.

The query represents the information that the user wants to retrieve from the indexed document.

In [19]:
# query = "What is qualitative data analysis?"
query = "What is Verbal Classification"


## 2. Query Embedding

The user query is converted into an embedding using the same embedding model used for the document chunks.

Using the same embedding model is important because both the query and the document chunks must exist in the same vector space for meaningful similarity comparison.

In [20]:
from src.embeddings.embedder import chunk_encoder

In [21]:
query_embedding = chunk_encoder([query])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Each embedding has 384 dimensions
Similarity matrix:
[[1.0000001]]


## 3. Similarity Search

The query embedding is passed to the vector store.

The vector store compares the query vector with the stored document vectors and retrieves the most semantically similar chunks.

The result of this stage is the relevant context required to answer the user's question.

In [22]:
# Querying the Vector store
# distance, indices = vectorStore.search(query_embedding, 3)
result = vectorStore.search(query_embedding, 3)

In [23]:
print(result[0])

[{'title': '**a) Understanding Verbal Classification**', 'breadcrumb': ['', '**Section-1** **Logical** **Reasoning**', '**Type-4**', '**Class 3: Verbal Classification** **in Logical Reasoning**', '**a) Understanding Verbal Classification**'], 'content': 'The essence of verbal classification is to recognize the intrinsic connection between words. This\nconnection could be based on a variety of factors, including but not limited to:\n\n\n✓ **Semantic Relationship:** Words may be related by their meaning, such as synonyms or antonyms.\n\n\n✓ **Functional Relationship:** Words may be related by their use or function.\n\n\n✓ **Part to Whole Relationship:** Words may have a relationship where one is a part of the other.\n\n\n✓ **Type or Category:** Words may belong to the same category or type, such as animals, tools, or\n\nemotions.\n\n\nThe outlier might not fit due to a difference in category, function, or some other distinguishing\ncharacteristic.', 'distance': 0.4487420916557312, '_fais

In [24]:
# # Preparing the Retrieved text List
# retrieved_text = []
# for idx in result:
#     retrieved_text.append(chunks[idx])

# retrieved_text

## 4. Prompt Construction and Response Generation

The retrieved document context and the original user query are combined to construct a prompt for the Large Language Model.

The LLM uses the retrieved context to generate the final response.

The final stage is:

User Query
        +
Retrieved Context
        ->
Prompt
        ->
Large Language Model
        ->
Generated Response

In [25]:
from src.generation.llm import LLMResponseGenerator

In [26]:
# LLM Configuration
llm =  LLMResponseGenerator()

In [27]:
context = llm.build_context(result[0])

# Prompt Building
prompt = llm.build_prompt(user_question=query, context=context)

In [28]:
# Response Generation
response = llm.generate_response(prompt=prompt)

In [29]:
print(response.text)

Verbal classification, also known as semantic classification, is a critical component of logical reasoning tests. It assesses an individual's ability to discern relationships between words based on their meanings, uses, or general characteristics (Source 2). The essence of verbal classification is to recognize the intrinsic connection between words (Source 1). This connection can be based on various factors, including semantic relationships (such as synonyms or antonyms), functional relationships (related by use or function), part to whole relationships, or belonging to the same type or category (such as animals, tools, or emotions) (Source 1).

Typically, these questions present a group of words, usually four, out of which three are related to each other in some way, while one is not. The challenge lies in identifying this outlier—the word that does not belong to the same classification as the others (Source 2).


# RAG Evaluation Dataset Generation

In [30]:
# from langchain_google_genai import ChatGoogleGenerativeAI

In [31]:
# API_KEY = 'AQ.Ab8RN6LdZl8pxdRhnCTAMsjCvQuzFv1CRJt3JGnqdkmhUR-GhQ'

In [32]:
# generator_llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0.2,
#     google_api_key=API_KEY,
# )

In [33]:
# from langchain_google_vertexai import ChatVertexAI, VertexAI

In [34]:
# import ragas

# print("Ragas:", ragas.__version__)

# from ragas.testset import TestsetGenerator

# print("TestsetGenerator import works")

In [35]:
# from langchain_community.chat_models.vertexai import ChatVertexAI


In [36]:
# from ragas.testset import TestsetGenerator
# from langchain_google_genai import ChatGoogleGenerativeAI


In [37]:
# from ragas.testset import TestsetGenerator
# from langchain_openai import ChatOpenAI

# # Use GPT-4o-mini or GPT-4 depending on your API access
# llm = ChatOpenAI(model="gpt-4o-mini")

# generator = TestsetGenerator(llm=llm)

# print("Setup OK:", generator)
